In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import os
import pathlib

# --- 1. Connect to Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    home = pathlib.Path.home()
    possible_drive_paths = [
        home / 'Google Drive' / 'My Drive' / 'CryptoProject',
        home / 'My Drive' / 'CryptoProject',
        pathlib.Path('G:/My Drive/CryptoProject'),
        pathlib.Path('G:/MyDrive/CryptoProject'),
        home / 'Google Drive' / 'MyDrive' / 'CryptoProject',
    ]
    BASE_DIR = None
    for p in possible_drive_paths:
        if p.parent.exists():
            BASE_DIR = str(p)
            print(f"Google Drive Desktop found! Saving to: {BASE_DIR}")
            break
    if BASE_DIR is None:
        print("Google Drive Desktop not found. Saving locally.")
        try:
            BASE_DIR = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            BASE_DIR = os.getcwd()

print(f"BASE_DIR = {BASE_DIR}")
SAVE_PATH = os.path.join(BASE_DIR, 'data')

BASE_URL = "https://api.binance.com/api/v3/klines"
COLUMNS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'number_of_trades',
    'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
]
NUMERIC_COLS = ['open', 'high', 'low', 'close', 'volume', 'quote_asset_volume',
                'number_of_trades', 'taker_buy_base_asset_volume',
                'taker_buy_quote_asset_volume']
INTERVAL_MS = {'m': 60_000, 'h': 3_600_000, 'd': 86_400_000}


class BinanceDataLoader:
    def __init__(self, symbol='BTCUSDT', interval='5m', save_path=SAVE_PATH):
        self.symbol = symbol
        self.interval = interval
        self.save_path = save_path
        if not os.path.exists(save_path):
            os.makedirs(save_path)
            print(f"Created directory at: {save_path}")

    def _interval_ms(self):
        return int(self.interval[:-1]) * INTERVAL_MS[self.interval[-1]]

    def _fetch_chunk(self, start_time, limit=1000):
        """Return a list of klines, [] at genuine end of data, or None on failure
        (after retries). Distinguishing None from [] is what makes the loop robust:
        a transient failure must NOT be mistaken for 'reached the end'."""
        params = {'symbol': self.symbol, 'interval': self.interval,
                  'startTime': start_time, 'limit': limit}
        max_retries = 6
        for attempt in range(max_retries):
            try:
                r = requests.get(BASE_URL, params=params, timeout=20)
                if r.status_code == 200:
                    return r.json()
                if r.status_code in (429, 418):          # rate limited / banned
                    wait = int(r.headers.get('Retry-After', 60))
                    print(f"   Rate limited ({r.status_code}); sleeping {wait}s...")
                    time.sleep(wait)
                    continue
                if 500 <= r.status_code < 600:            # transient server error
                    print(f"   Server error {r.status_code}; retry {attempt+1}/{max_retries}")
                    time.sleep(min(60, 2 ** attempt))
                    continue
                print(f"   Error {r.status_code}: {r.text[:200]}")
                return None
            except requests.exceptions.RequestException as e:
                print(f"   Connection error (try {attempt+1}/{max_retries}): {e}")
                time.sleep(min(60, 2 ** attempt))
        return None

    def _typed(self, df):
        if df.empty:
            return df
        df = df.copy()
        for col in NUMERIC_COLS:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
        df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')
        return df

    def _merge_and_save(self, existing, all_data):
        new_df = self._typed(pd.DataFrame(all_data, columns=COLUMNS)) if all_data \
            else pd.DataFrame(columns=COLUMNS)
        frames = [d for d in (existing, new_df) if d is not None and not d.empty]
        df = pd.concat(frames, ignore_index=True) if frames else new_df
        if not df.empty:
            df = (df.drop_duplicates(subset=['open_time'], keep='last')
                    .sort_values('open_time').reset_index(drop=True))
        self._save_to_csv(df)
        return df

    def fetch_history(self, start_str, end_str=None, resume=True):
        """Download [start, end] klines, paginating forward. Resumes from an
        existing CSV by default, retries transient failures, checkpoints
        periodically, and only stops when the API genuinely returns no more data.
        Set resume=False to re-download from scratch."""
        print(f"--- Collecting {self.symbol} ({self.interval}) ---")
        start_ts = int(pd.Timestamp(start_str, tz='UTC').timestamp() * 1000)
        end_ts = int(pd.Timestamp(end_str, tz='UTC').timestamp() * 1000) if end_str \
            else int(time.time() * 1000)

        filename = os.path.join(self.save_path, f"{self.symbol}_{self.interval}_data.csv")
        existing = None
        current_start = start_ts
        if resume and os.path.exists(filename):
            try:
                existing = pd.read_csv(filename)
                existing['open_time'] = pd.to_datetime(existing['open_time'])
                if not existing.empty:
                    last_open = int(existing['open_time'].iloc[-1].tz_localize('UTC').timestamp() * 1000)
                    current_start = max(start_ts, last_open + 1)
                    print(f"Resuming from {filename} ({len(existing)} rows; "
                          f"last candle {existing['open_time'].iloc[-1]}).")
            except Exception as e:
                print(f"Could not read existing CSV ({e}); starting fresh.")
                existing = None

        all_data = []
        step = self._interval_ms()
        fails = 0
        last_save_len = 0

        while current_start < end_ts:
            chunk = self._fetch_chunk(current_start)

            if chunk is None:                       # fetch failed after retries
                fails += 1
                if fails >= 3:
                    self._merge_and_save(existing, all_data)   # keep progress
                    raise RuntimeError(
                        f"Aborting near {datetime.fromtimestamp(current_start/1000, tz=timezone.utc)} "
                        f"after repeated failures. Progress saved to {filename} — "
                        f"just re-run this cell to resume.")
                continue
            fails = 0

            if len(chunk) == 0:                     # genuine end of available data
                print("Reached the most recent available candle.")
                break

            all_data.extend(chunk)
            nxt = chunk[-1][6] + 1                   # last close_time + 1ms
            current_start = nxt if nxt > current_start else current_start + step

            last_dt = datetime.fromtimestamp(chunk[-1][0] / 1000, tz=timezone.utc)
            if len(all_data) - last_save_len >= 100_000:    # periodic checkpoint
                self._merge_and_save(existing, all_data)
                last_save_len = len(all_data)
                print(f"   checkpoint: {len(all_data)} new rows (up to {last_dt})")
            else:
                print(f"Fetched up to {last_dt}  (+{len(all_data)} new rows)")
            time.sleep(0.1)

        df = self._merge_and_save(existing, all_data)
        if not df.empty:
            span = df['open_time'].max()
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            gap_days = (now - span).total_seconds() / 86400
            print(f"--- Done. {len(df)} rows, {df['open_time'].min()} -> {span}. ---")
            if gap_days > 1:
                print(f"WARNING: newest candle is {gap_days:.1f} days old — "
                      f"download may be incomplete; re-run to resume.")
        return df

    def _save_to_csv(self, df):
        filename = os.path.join(self.save_path, f"{self.symbol}_{self.interval}_data.csv")
        df.to_csv(filename, index=False)


if __name__ == "__main__":
    SYMBOLS = [
        ('BTCUSDT', '2017-09-09'),
        ('ETHUSDT', '2017-09-09'),
        ('XRPUSDT', '2017-09-09'),
    ]
    for symbol, start_date in SYMBOLS:
        print(f"\n{'='*50}\nProcessing: {symbol}\n{'='*50}")
        loader = BinanceDataLoader(symbol=symbol, interval='5m')
        # resume=True continues an existing/partial CSV (e.g. one that stopped in
        # 2019). To force a clean re-download from scratch, pass resume=False.
        df = loader.fetch_history(start_str=start_date, end_str=None, resume=True)
        expected = os.path.join(SAVE_PATH, f"{symbol}_{loader.interval}_data.csv")
        print("Success!" if os.path.exists(expected) else "Error: file not created",
              "->", expected)